In [1]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO)
from rexpand_pyutils_file import write_file

from utils.conversation_data import load_conversation_data
from utils.json import to_json_compatible
from models.context import Context
from nodes.orchestrator import orchestrate
from models.workflow import State


INFO:root:LLM_MODEL: gpt-4.1-mini
INFO:root:LLM_TEMPERATURE: 0.0
INFO:root:LLM_USE_CACHE: False
INFO:root:LLM_INCLUDE_DEBUG_FIELDS: False
INFO:root:Read JSON data from ./input/categories.json
INFO:root:Read JSON data from ./input/categories.json


In [2]:
# Load all conversations
conversations = load_conversation_data('./input/convo_2454_rows.xlsx')


INFO:root:Read sheet 'Result 1' from Excel file ./input/convo_2454_rows.xlsx
INFO:root:Read EXCEL data from ./input/convo_2454_rows.xlsx with 2454 rows


In [3]:
# Select a conversation to process
index = 2
messages_end = 4

# In this case, the referrer doesn't reply
messages = conversations[index][:messages_end]

context = Context(messages=messages)
write_file('./output/current_messages.json', to_json_compatible(messages))

state = State(context=context)
write_file('./output/state.json', { "state": to_json_compatible(state.model_dump())})


INFO:root:Wrote JSON data to ./output/current_messages.json
INFO:root:Wrote JSON data to ./output/state.json


In [4]:
# Fresh start
state = orchestrate(state)

# In this case, no user action is required and message will be directly
#  generated

print(state.step)
print(state.classified_category)
print(state.suggested_topics)
print(state.selected_topics)
print(state.generated_reply_message)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


end: reply generated
ClassifierResult(category='no_reply', confidence=None, reason=None, referenced_message_ids=['fcf62787-5474-4ba4-a04d-0f7c5f9e9a5c'])
TopicSuggesterResult(topics=[Topic(topic='Thank you', confidence=None, reason=None), Topic(topic='Express interest', confidence=None, reason=None), Topic(topic='Ask for a brief call', confidence=None, reason=None), Topic(topic='Express background match', confidence=None, reason=None)])
['Thank you', 'Express interest', 'Ask for a brief call', 'Express background match']
MessageGeneratorResult(message='Hi Michael,\n\nThanks for getting back to me and for considering a referral. I’m very interested in EY’s Junior Data Analyst role and believe my background at The University of Chicago aligns with the position. I’d greatly appreciate any guidance you can offer on the referral and the application process. Would you have 15 minutes for a quick call this week to discuss? If a call isn’t convenient, please let me know the best next steps or 

In [5]:
print(state.generated_reply_message.message)


Hi Michael,

Thanks for getting back to me and for considering a referral. I’m very interested in EY’s Junior Data Analyst role and believe my background at The University of Chicago aligns with the position. I’d greatly appreciate any guidance you can offer on the referral and the application process. Would you have 15 minutes for a quick call this week to discuss? If a call isn’t convenient, please let me know the best next steps or how you’d prefer to proceed.

Thank you again for your time.

Best,
Patty
